## 💬 Product Check-in: API Existence Checks

> **Sam (Product):** "Bigger issue — users are asking whether specific APIs exist in MLflow and the agent is confidently making things up. Someone asked about `mlflow.genai.evaluate` and the agent described a completely different function signature. They wasted an hour debugging code that was never going to work."
>
> **You:** "Yeah, that's a hallucination problem. The LLM has seen a lot of MLflow docs in training but it pattern-matches on what *sounds* right rather than what *is* right. It'll fabricate parameter names with complete confidence."
>
> **Sam:** "Can we do anything without a full doc retrieval system? That feels heavy for this sprint."
>
> **You:** "We can get most of the way there with a simple lookup tool. A hardcoded dictionary of the most commonly asked-about APIs — function name, parameters, a one-line description. The agent calls it instead of guessing. Not exhaustive, but it covers the high-traffic cases and stops the worst hallucinations."
>
> **Sam:** "How do we know it's using the lookup and not still guessing?"
>
> **You:** "Same as before — `ToolCallCorrectness` and `ToolCallEfficiency` in the eval pipeline. I'll also add eval cases where the expected tool call is explicitly specified so we can catch regressions."
>

---

Let's do exactly that. We'll:
1. Build the `check_api_exists` tool
2. Add it to the agent
3. Add `ToolCallCorrectness` and `ToolCallEfficiency` to the eval pipeline
4. Write eval cases with `expected_tool_calls` ground truth

```python
# These examples should FAIL before the tool is added —
# the bare LLM will hallucinate parameter names or confidently describe wrong signatures

```

In [ ]:
api_eval_dataset = [
    {
        "inputs": {"question": "Does mlflow.genai.evaluate exist in MLflow?"},
        "expectations": {
            "expected_facts": ["yes", "mlflow.genai.evaluate"],
            "guidelines": [
                "Response must confirm the API exists, not speculate",
                "Response must not fabricate parameter names not in the actual signature",
            ],
        },
    },
    {
        "inputs": {"question": "What parameters does mlflow.genai.evaluate take?"},
        "expectations": {
            "expected_facts": ["data", "scorers"],
            "guidelines": [
                "Response must only reference parameter names that exist in the actual API",
                "Response must not describe the pre-3.0 mlflow.evaluate() signature",
            ],
        },
    },
    {
        "inputs": {"question": "Is there a make_judge function in MLflow?"},
        "expectations": {
            "expected_facts": ["yes", "make_judge"],
            "guidelines": [
                "Response must confirm the function exists rather than guessing",
                "Response must not conflate it with a different MLflow API",
            ],
        },
    },
]
print(f"API eval set size: {len(api_eval_dataset)} examples")